# 03 · Variables por cliente (RFM extendido)

> **Objetivo:** construir el DataFrame por cliente que la Unidad 3 usa
> como entrada. La funcion `construir_rfm(ventas, fecha_corte)` es la
> misma que U3 reutiliza con cortes anteriores; aqui la llamamos con
> `FECHA_SNAPSHOT = max(InvoiceDate) + 1 dia`.

## Variables que vamos a dejar

- **recencia:** dias desde la ultima compra hasta el corte.
- **antiguedad:** dias desde la primera compra hasta el corte.
- **frecuencia:** facturas distintas.
- **monetario:** suma de `Importe`.
- **productos_distintos:** `StockCode` unicos.

In [1]:
# Permite importar el paquete segmentacion desde el notebook
import sys
from pathlib import Path

_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from datetime import timedelta

import polars as pl
from segmentacion.config import PROCESSED_DATA_DIR
from segmentacion.data.cargar import leer_parquet
from segmentacion.features.rfm import construir_rfm

PARQUET_VENTAS = PROCESSED_DATA_DIR / "ventas_validas.parquet"
PARQUET_RFM = PROCESSED_DATA_DIR / "rfm_u2.parquet"
assert PARQUET_VENTAS.exists(), "Ejecuta `02_limpieza.ipynb` primero."
ventas = leer_parquet(PARQUET_VENTAS)
FECHA_SNAPSHOT = ventas["InvoiceDate"].max() + timedelta(days=1)
print(f"FECHA_SNAPSHOT: {FECHA_SNAPSHOT}")
print(f"ventas: {ventas.height:,}".replace(",", " "))

FECHA_SNAPSHOT: 2011-12-10 12:50:00
ventas: 779 425


## 1. Construir RFM

In [3]:
rfm = construir_rfm(ventas, FECHA_SNAPSHOT)
print(f"clientes: {rfm.height:,}".replace(",", " "))
rfm.head()

clientes: 5 878


Customer ID,recencia,antiguedad,frecuencia,monetario,productos_distintos
i64,i64,i64,u32,f64,u32
12346,326,726,12,77556.46,27
12347,2,404,8,4921.53,126
12348,75,438,5,2019.4,25
12349,19,589,4,4428.69,138
12350,310,310,1,334.4,17


## 2. Distribuciones

Cola larga en `monetario` y `frecuencia`: ambos son candidatos a `log1p`
en U3.

In [4]:
rfm.select(["recencia", "antiguedad", "frecuencia", "monetario", "productos_distintos"]).describe()

statistic,recencia,antiguedad,frecuencia,monetario,productos_distintos
str,f64,f64,f64,f64,f64
"""count""",5878.0,5878.0,5878.0,5878.0,5878.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",201.331916,474.711637,6.289384,2955.904095,81.989112
"""std""",209.338707,223.098342,13.009406,14440.852688,116.484552
"""min""",1.0,1.0,1.0,2.95,1.0
"""25%""",26.0,313.0,1.0,342.24,19.0
"""50%""",96.0,530.0,3.0,868.37,45.0
"""75%""",380.0,668.0,7.0,2248.5,103.0
"""max""",739.0,739.0,398.0,580987.04,2550.0


## 3. Asociacion frecuencia-monetario (comprobacion rapida)

In [ ]:
rho = rfm.select(pl.corr("frecuencia", "monetario", method="spearman")).item()
print(f"Spearman frecuencia-monetario: {rho:.3f}  (ref: 0,858)")

## 4. Guardar para la Unidad 3

In [6]:
rfm.write_parquet(PARQUET_RFM, compression="zstd")
print(f"escrito: {PARQUET_RFM} ({PARQUET_RFM.stat().st_size / 1e6:.1f} MB)")

escrito: /Users/david.palacio/Documents/data-projects-lab/projects/00-segmentacion-clientes/data/processed/rfm_u2.parquet (0.1 MB)


## Preguntas de reflexion

1. `construir_rfm` usa `InvoiceDate < fecha_corte` (estrictamente menor).
   ¿Que riesgo evita ese `<` cuando U3 defina cortes para evaluar
   recompra 90 dias en el futuro?
2. ¿Deberia `antiguedad` entrar al pipeline sklearn de U3 como feature
   crudo, o tiene sentido convertirlo a razon (`antiguedad / 365`)?
3. ¿Que indica la fuerte Spearman frecuencia-monetario para la
   seleccion de features? ¿Podrian estar midiendo lo mismo?

> **Siguiente paso:** `05_eda_automatizado.ipynb` — perfil con
> `skimpy` y `fg-data-profiling` sobre una muestra.